# Эксперимент: OpenL3 + SVM

**Модель:** OpenL3 эмбеддинги (audio-audio) → агрегация mean+max → SVM RBF  
**Группа:** 03_pretrained_frozen  
**Ориентир:** checkpoint_3/exp_18 — Acc=0.779, F1-bad=0.664, ROC-AUC=0.817

## Исправления

1. **5-fold CV** с оптимизацией порога.
2. **Holdout test изолирован** до обучения.

In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
import openl3
import soundfile as sf
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

In [6]:
def extract_openl3(path: str) -> np.ndarray:
    """OpenL3 audio-audio → mean+max (1024-dim)."""
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    emb, ts = openl3.get_audio_embedding(
        audio, sr,
        content_type="music",
        embedding_size=512,
        hop_size=0.5,
        verbose=False,
    )
    return np.concatenate([emb.mean(axis=0), emb.max(axis=0)]).astype(np.float32)

In [7]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()
print(f"Train+Val: {len(paths_trainval)}  |  Holdout Test: {len(paths_test)}")

Train+Val: 2359  |  Holdout Test: 417


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.01, 0.1],
}

with start_run("exp_openl3_svm", group="03_pretrained_frozen"):

    mlflow.log_params({
        "encoder":        "openl3 audio-audio",
        "encoder_frozen": True,
        "embedding_size": 512,
        "aggregation":    "mean+max",
        "embed_dim":      1024,
        "classifier":     "SVM RBF",
        "cv_folds":       config.CV_N_SPLITS,
        "class_weight":   "balanced",
    })

    fold_results = []
    t0 = time.perf_counter()

    for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
            get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

        print(f"\nФолд {fold+1}/{config.CV_N_SPLITS}")
        X_tr  = build_feature_matrix(paths_tr,  extract_openl3, n_jobs=4)
        X_val = build_feature_matrix(paths_val, extract_openl3, n_jobs=4)
        X_tr  = np.hstack([X_tr,  letters_tr])
        X_val = np.hstack([X_val, letters_val])

        grid = GridSearchCV(pipeline, param_grid, cv=3,
                            scoring="f1_macro", refit=True, n_jobs=-1)
        grid.fit(X_tr, labels_tr)
        clf = grid.best_estimator_
        print(f"  best={grid.best_params_}")

        y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
        thr = find_optimal_threshold(labels_val, y_proba_val)
        metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=True)
        fold_results.append(metrics)
        log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                    roc_auc=metrics["roc_auc"], threshold=thr)

    train_time_sec = time.perf_counter() - t0
    cv_agg = evaluate_cv_folds(fold_results)
    print_cv_summary(cv_agg)

    print("\nФинальная модель на train+val...")
    X_trainval = build_feature_matrix(paths_trainval, extract_openl3, n_jobs=4)
    X_test     = build_feature_matrix(paths_test,     extract_openl3, n_jobs=4)
    X_trainval = np.hstack([X_trainval, letters_trainval])
    X_test     = np.hstack([X_test,     letters_test])

    final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                              scoring="f1_macro", refit=True, n_jobs=-1)
    final_grid.fit(X_trainval, labels_trainval)
    mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

    cv_threshold = cv_agg["threshold_mean"]
    y_proba_test = final_grid.best_estimator_.predict_proba(X_test)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)

    save_result_csv(
        exp_dir=exp_dir,
        experiment_id="exp_openl3_svm",
        experiment_name="OpenL3 + SVM",
        model="OpenL3 mean+max + SVM RBF",
        accuracy=test_metrics["accuracy"],
        f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],
        roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"],
        recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        cv_f1_bad_std=cv_agg["f1_bad_std"],
        cv_f1_macro_std=cv_agg["f1_macro_std"],
        notes=f"5-fold CV + holdout | best={final_grid.best_params_} | thr={cv_threshold:.2f} | embed=512 mean+max",
        train_time_sec=train_time_sec,
    )
    print("Результаты сохранены")

df_folds = pd.DataFrame(fold_results)
df_folds.index = [f"Fold {i+1}" for i in range(len(fold_results))]
display(df_folds[["accuracy", "f1_macro", "f1_bad", "roc_auc", "threshold"]])


Фолд 1/5


/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin 

  best={'clf__C': 1.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.85      0.81      0.83       318
         bad       0.64      0.69      0.67       154

    accuracy                           0.77       472
   macro avg       0.74      0.75      0.75       472
weighted avg       0.78      0.77      0.78       472

Threshold : 0.38
Accuracy  : 0.7733
F1-macro  : 0.7475
F1-bad    : 0.6667
ROC-AUC   : 0.8108

Фолд 2/5


/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/dk/miniconda3/envs/vkr/lib/python3.10/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin 

KeyboardInterrupt: 